# Joint Supervised-SSL Model Experiment
### for Assignment 2: SSL
**By Tahiyat Chy**

This model combines the main MAE model with the Supervised baseline to test whether it can achieve higher accuracy than

## Initial Setup

In [1]:
# Core imports
from pathlib import Path
import random
import time
import copy

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets, transforms

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.11.0+cu130
CUDA available: True


In [2]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except ImportError:
    print("Not running in Google Colab; Drive mount skipped.")

Mounted at /content/gdrive


In [3]:
# Path set to assignment directory

PROJECT_ROOT = Path('/content/gdrive/MyDrive/ITCS_6169/assignment2')
DATA_ROOT = PROJECT_ROOT / 'data'
TRAIN_ROOT = DATA_ROOT / 'train'
TEST_ROOT = DATA_ROOT / 'test'

print('Project root:', PROJECT_ROOT.resolve())
print('Training data:', TRAIN_ROOT)
print('Testing data:', TEST_ROOT)

Project root: /content/gdrive/MyDrive/ITCS_6169/assignment2
Training data: /content/gdrive/MyDrive/ITCS_6169/assignment2/data/train
Testing data: /content/gdrive/MyDrive/ITCS_6169/assignment2/data/test


In [4]:
SEED = 0

def set_random_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_random_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


## Data Preparation
Define data augmentations. Split the training images into training and validation set.

In [5]:
import torchvision.transforms as T

BATCH_SIZE = 64
IMG_SIZE = 224
NUM_CLASSES = 16

# Data augmentation
train_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandAugment(),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.1)
])

val_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
full_train_dataset = datasets.ImageFolder(root=TRAIN_ROOT)

# Split into train and val
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size])

# Implement custom wrapper to apply different transforms to train and val subsets
class TransformSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

train_dataset = TransformSubset(train_subset, transform=train_transforms)
val_dataset = TransformSubset(val_subset, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Total training images: {len(train_dataset)}")
print(f"Total validation images: {len(val_dataset)}")

Total training images: 1920
Total validation images: 480


## Model Architecture


In [6]:
class ConvPatchEmbed(nn.Module):
    """Convolutional Patch Projection for Vision Transformers."""
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=192):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        # Patch projection using standard Conv2d
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # Input: (B, C, H, W) -> Projection: (B, embed_dim, H/p, W/p) -> Tokens: (B, N, embed_dim)
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

In [7]:
class JointViTMAE(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_channels=3,
        num_classes=16,
        embed_dim=192,       # Downscaled representation to avoid overfitting
        depth=6,
        num_heads=4,
        decoder_embed_dim=128,
        decoder_depth=2,
        decoder_num_heads=4
    ):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_dim = in_channels * patch_size * patch_size

        # 1. Convolutional Patch Projection
        self.patch_embed = ConvPatchEmbed(img_size, patch_size, in_channels, embed_dim)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))

        # 2. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4,
            batch_first=True, activation='gelu', dropout=0.1
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        # 3. Classification Head
        self.head = nn.Linear(embed_dim, num_classes)

        # 4. Lightweight MAE Decoder
        self.decoder_embed = nn.Linear(embed_dim, decoder_embed_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, decoder_embed_dim))

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=decoder_embed_dim, nhead=decoder_num_heads, dim_feedforward=decoder_embed_dim * 4,
            batch_first=True, activation='gelu'
        )
        self.decoder = nn.TransformerEncoder(decoder_layer, num_layers=decoder_depth)
        self.decoder_pred = nn.Linear(decoder_embed_dim, self.patch_dim)

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.decoder_pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.mask_token, std=0.02)

    def patchify(self, x):
        """Converts raw image pixels into flattened patch vectors for MSE target computation."""
        B, C, H, W = x.shape
        p = self.patch_size
        patches = x.view(B, C, H // p, p, W // p, p)
        patches = patches.permute(0, 2, 4, 3, 5, 1).contiguous()
        return patches.view(B, self.num_patches, self.patch_dim)

    def forward_encoder(self, x, mask_ratio):
        B = x.shape[0]
        patches = self.patchify(x)

        x_embed = self.patch_embed(x) + self.pos_embed[:, 1:, :]

        if mask_ratio > 0:
            len_keep = int(self.num_patches * (1 - mask_ratio))
            noise = torch.rand(B, self.num_patches, device=x.device)
            ids_shuffle = torch.argsort(noise, dim=1)
            ids_restore = torch.argsort(ids_shuffle, dim=1)
            ids_keep = ids_shuffle[:, :len_keep]

            x_masked = torch.gather(x_embed, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, x_embed.shape[-1]))
        else:
            x_masked = x_embed
            ids_restore = None

        cls_token = self.cls_token + self.pos_embed[:, :1, :]
        x_enc_in = torch.cat((cls_token.expand(B, -1, -1), x_masked), dim=1)
        x_enc_out = self.encoder(x_enc_in)
        return x_enc_out, patches, ids_restore

    def forward_decoder(self, x_enc_out, ids_restore):
        B = x_enc_out.shape[0]
        cls_tok = x_enc_out[:, :1, :]
        latent = x_enc_out[:, 1:, :]

        latent_dec = self.decoder_embed(latent)
        len_keep = latent_dec.shape[1]

        mask_tokens = self.mask_token.repeat(B, self.num_patches - len_keep, 1)
        x_full = torch.cat([latent_dec, mask_tokens], dim=1)
        x_full = torch.gather(x_full, dim=1, index=ids_restore.unsqueeze(-1).repeat(1, 1, x_full.shape[-1]))

        x_full = x_full + self.decoder_pos_embed[:, 1:, :]
        cls_dec = self.decoder_embed(cls_tok) + self.decoder_pos_embed[:, :1, :]

        x_dec_out = self.decoder(torch.cat([cls_dec, x_full], dim=1))
        return self.decoder_pred(x_dec_out[:, 1:, :])

    def forward(self, x, mask_ratio=0.75):
        if self.training and mask_ratio > 0:
            x_enc_out, patches, ids_restore = self.forward_encoder(x, mask_ratio)
            cls_logits = self.head(x_enc_out[:, 0])
            pred_patches = self.forward_decoder(x_enc_out, ids_restore)
            return cls_logits, pred_patches, patches, ids_restore
        else:
            x_enc_out, _, _ = self.forward_encoder(x, mask_ratio=0.0)
            return self.head(x_enc_out[:, 0])

In [8]:
def compute_mae_loss(pred_patches, target_patches, ids_restore, mask_ratio=0.75):
    """Computes MSE reconstruction loss strictly across masked patches."""
    B, N, _ = pred_patches.shape
    len_keep = int(N * (1 - mask_ratio))

    # Calculate MSE per patch
    loss_patch = ((pred_patches - target_patches) ** 2).mean(dim=-1)

    # Generate binary mask matrix (0 = kept, 1 = masked)
    mask = torch.ones([B, N], device=pred_patches.device)
    mask[:, :len_keep] = 0
    mask = torch.gather(mask, dim=1, index=ids_restore)

    # Mean loss calculated strictly on masked patches
    return (loss_patch * mask).sum() / mask.sum()

## Training Loop

In [9]:
def train_joint_model(model, train_loader, val_loader, epochs=30, lr=5e-4, lambda_recon=1.0, mask_ratio=0.75):
    cls_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        running_loss, running_cls_loss, running_recon_loss = 0.0, 0.0, 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            cls_logits, pred_patches, target_patches, ids_restore = model(inputs, mask_ratio=mask_ratio)

            loss_cls = cls_criterion(cls_logits, labels)
            loss_recon = compute_mae_loss(pred_patches, target_patches, ids_restore, mask_ratio)

            # Combined Loss Objective
            loss = loss_cls + lambda_recon * loss_recon

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_cls_loss += loss_cls.item() * inputs.size(0)
            running_recon_loss += loss_recon.item() * inputs.size(0)

        scheduler.step()

        total_samples = len(train_loader.dataset)
        epoch_loss = running_loss / total_samples
        epoch_cls = running_cls_loss / total_samples
        epoch_recon = running_recon_loss / total_samples

        # Validation Step (Full images without masking)
        model.eval()
        corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                corrects += torch.sum(preds == labels.data)

        epoch_acc = corrects.double() / len(val_loader.dataset)

        print(f"Epoch {epoch+1:02d}/{epochs} | Total Loss: {epoch_loss:.4f} (Cls: {epoch_cls:.4f}, Recon: {epoch_recon:.4f}) | Val Acc: {epoch_acc:.4f}")

        if epoch_acc > best_acc:
            best_acc = epoch_acc
            best_model_wts = copy.deepcopy(model.state_dict())

    print(f"\nTraining complete. Best Validation Accuracy: {best_acc:.4f}")
    model.load_state_dict(best_model_wts)
    return model

In [10]:
# Instantiate model
model = JointViTMAE(num_classes=NUM_CLASSES).to(device)
print(f"Total Model Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Execute training pass
joint_model = train_joint_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=100,
    lr=5e-4,
    lambda_recon=1.0,
    mask_ratio=0.75
)

Total Model Parameters: 3,403,600
Epoch 01/100 | Total Loss: 4.1525 (Cls: 2.7782, Recon: 1.3743) | Val Acc: 0.1271
Epoch 02/100 | Total Loss: 3.8635 (Cls: 2.6631, Recon: 1.2004) | Val Acc: 0.1646
Epoch 03/100 | Total Loss: 3.7731 (Cls: 2.6252, Recon: 1.1480) | Val Acc: 0.1979
Epoch 04/100 | Total Loss: 3.6965 (Cls: 2.5894, Recon: 1.1071) | Val Acc: 0.2062
Epoch 05/100 | Total Loss: 3.7154 (Cls: 2.5968, Recon: 1.1186) | Val Acc: 0.1938
Epoch 06/100 | Total Loss: 3.6760 (Cls: 2.5810, Recon: 1.0949) | Val Acc: 0.1917
Epoch 07/100 | Total Loss: 3.6521 (Cls: 2.5147, Recon: 1.1373) | Val Acc: 0.2625
Epoch 08/100 | Total Loss: 3.5728 (Cls: 2.4475, Recon: 1.1253) | Val Acc: 0.2354
Epoch 09/100 | Total Loss: 3.5020 (Cls: 2.4109, Recon: 1.0911) | Val Acc: 0.2333
Epoch 10/100 | Total Loss: 3.4628 (Cls: 2.3687, Recon: 1.0941) | Val Acc: 0.3000
Epoch 11/100 | Total Loss: 3.4280 (Cls: 2.3652, Recon: 1.0628) | Val Acc: 0.2708
Epoch 12/100 | Total Loss: 3.3708 (Cls: 2.3358, Recon: 1.0349) | Val Acc: 0